# PostFab BGE-M3 파인튜닝

**순서**: 1번 셀 실행 → 커널 재시작 → 2번부터 순서대로 실행

In [ ]:
# 1. 패키지 설치 (실행 후 런타임 재시작)
!pip install -q --force-reinstall torch torchvision sentence-transformers==3.4.1 transformers==4.49.0 huggingface_hub==0.30.2 datasets
import os
os.kill(os.getpid(), 9)

In [ ]:
# 2. Drive 마운트 및 파일 확인
from google.colab import drive
import os

drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/PostFab'
os.chdir(DATA_DIR)

for f in ['train.json', 'test.json', 'ir_eval.json']:
    print(f'{f}: {"✅" if os.path.exists(f) else "❌"}')

In [ ]:
# 3. GPU 확인
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# 4. Before 평가
import json
from sentence_transformers import SentenceTransformer
from sentence_transformers.evaluation import InformationRetrievalEvaluator

with open('ir_eval.json', encoding='utf-8') as f:
    ir = json.load(f)

evaluator = InformationRetrievalEvaluator(
    queries=ir['queries'],
    corpus=ir['corpus'],
    relevant_docs={k: set(v) for k, v in ir['relevant_docs'].items()},
    name='postfab-test',
    accuracy_at_k=[1, 3, 5],
    ndcg_at_k=[10],
    show_progress_bar=True
)

model_before = SentenceTransformer('BAAI/bge-m3')
results_before = evaluator(model_before)

print('\n=== Before 파인튜닝 ===')
print(f'Accuracy@1:  {results_before["postfab-test_cosine_accuracy@1"]:.4f}')
print(f'Accuracy@3:  {results_before["postfab-test_cosine_accuracy@3"]:.4f}')
print(f'Accuracy@5:  {results_before["postfab-test_cosine_accuracy@5"]:.4f}')
print(f'NDCG@10:     {results_before["postfab-test_cosine_ndcg@10"]:.4f}')

In [ ]:
# 5. 파인튜닝
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import MultipleNegativesRankingLoss
from datasets import Dataset
import os

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

with open('train.json', encoding='utf-8') as f:
    train_dataset = Dataset.from_list(json.load(f))

model = SentenceTransformer('BAAI/bge-m3')
loss = MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir='postfab-bge-m3',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    warmup_ratio=0.1,
    learning_rate=2e-5,
    fp16=True,
    save_strategy='epoch',
    logging_steps=10,
    load_best_model_at_end=False,
    report_to='none',
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=loss,
    evaluator=evaluator,
)

trainer.train()

In [ ]:
# 6. After 평가
results_after = evaluator(model)

print('\n========== 결과 비교 ==========')
metrics = [
    ('Accuracy@1', 'postfab-test_cosine_accuracy@1'),
    ('Accuracy@3', 'postfab-test_cosine_accuracy@3'),
    ('Accuracy@5', 'postfab-test_cosine_accuracy@5'),
    ('NDCG@10',    'postfab-test_cosine_ndcg@10'),
]
for label, key in metrics:
    before = results_before[key]
    after = results_after[key]
    diff = after - before
    arrow = '▲' if diff > 0 else '▼'
    print(f'{label:15s}: {before:.4f} → {after:.4f}  {arrow} {abs(diff):.4f}')

In [ ]:
# 7. 모델 저장 및 다운로드
from google.colab import files

model.save('postfab-bge-m3-final')
!zip -r postfab-bge-m3-final.zip postfab-bge-m3-final/
files.download('postfab-bge-m3-final.zip')